# 📚 03_统计分析与展示

> **核心能力**：分组统计 (groupby)、排序排名、文本提取、时间切片、NumPy 标签
>
> **开局心法**：`groupby` 是 SQL GROUP BY 的镜像；`transform` 能把结果广播回原表；`.str` 和 `.dt` 是文本和时间的瑞士军刀；`np.where/select` 是打标签的终极武器。

---

## 1️⃣ 分组统计：GroupBy（SQL GROUP BY 的镜像）

> **大白话口诀**：这是纯正的 SQL `GROUP BY` 的镜像。先分组，再在每组里求和/平均。

In [ ]:
import pandas as pd
import numpy as np

# 【SQL 视角】
# SELECT 季度, SUM(销量) AS 汇总销量 
# FROM df 
# GROUP BY 季度 
# ORDER BY 汇总销量 ASC;

# 【Pandas 镜像翻译】
result = df.groupby('季度')['销量'].sum().sort_values(ascending=True)

# 或者更详细的写法
result = df.groupby('季度').agg({'销量': 'sum'}).sort_values(by='销量', ascending=True)

In [ ]:
# 【更多 groupby 例子】

# 1. 🔢 [多列分组 + 多个统计]：按"品牌"和"季度"分组，同时求均价和总销量
summary = df.groupby(['品牌', '季度']).agg({
    '价格': 'mean',        # 均价
    '销量': 'sum'         # 总销量
})

# 2. 📊 [自定义聚合函数]：比如求最大值和最小值的差
df.groupby('品牌')['价格'].agg(['min', 'max', lambda x: x.max() - x.min()])

# 3. 🚀 [行转列展示]：reset_index() 把索引变回普通列，方便后续操作
summary_reset = summary.reset_index()

### 🚨 坑点：groupby 结果的索引陷阱

#### 坑点 1：groupby 后的结果被当成了索引，而不是普通列

```python
# ❌ 常见错误
result = df.groupby('品牌')['销量'].sum()
# 此时 result 的"品牌"不是列，而是黑粗体的索引！
# 后续想用 result['品牌'] 会报错

# ✅ 解决方案：reset_index()
result = df.groupby('品牌')['销量'].sum().reset_index()
# 现在 result 有两列：'品牌' 和 '销量'（都是普通列）
```

---

#### 坑点 2：groupby 后无法直接和其他表 merge

```python
# ❌ 因为"品牌"是索引，merge 时出错
grouped = df.groupby('品牌')['销量'].sum()
result = pd.merge(grouped, other_df, on='品牌')  # 报错！

# ✅ 先 reset_index
grouped = df.groupby('品牌')['销量'].sum().reset_index()
result = pd.merge(grouped, other_df, on='品牌')  # ✓
```

---
## 2️⃣ 排序与排名

### 【快速操作】找销售额最高的 Top 3 品牌

```python
top_3 = df.groupby('品牌')['销售额'].sum().sort_values(ascending=False).head(3)
```

In [ ]:
# 1. 📊 [按某列排序]：找"销售额"最高的前 10 行
top_10 = df.sort_values('销售额', ascending=False).head(10)

# 2. 🏅 [分组内排名]：在每个"品牌"内，按"价格"排名，看谁最贵
df['组内排名'] = df.groupby('品牌')['价格'].rank(method='dense', ascending=False)
# method='dense' 表示：1,1,3（跳过 2），不是 1,1,2
# method='first' 表示：1,2,3（有重复就给不同排名，按出现顺序）

# 3. 🚀 [找各品牌最贵的那一款]：
most_expensive = df.loc[df.groupby('品牌')['价格'].idxmax()]
# idxmax() 返回每组里最大值所在的索引位置

---
## 3️⃣ 窗口函数与数据广播（transform）

> **大白话口诀**：
> 不要把原表"折叠/弄短"，而是在最后面**加一列全局或者组内统计值**，好让当前行直接跟全局去比（比如：我比平均高多少？）。

这对应 SQL 的 `PARTITION BY` 窗口函数。

In [ ]:
# 1. 🌍 [全表统计广播]：每一行都显示"全市场总销售额"
df['全市场销售额'] = df['销售额'].sum()
df['市场占比'] = (df['销售额'] / df['全市场销售额'] * 100).round(2)

# 2. 👥 [分组统计广播]：每行显示"自己所在品牌的平均价格"
# transform 的魔力就在这里：返回的结果长度和原表一样！
df['本品牌均价'] = df.groupby('品牌')['价格'].transform('mean')
df['我比品牌均价高多少'] = df['价格'] - df['本品牌均价']

# 3. 🎖️ [分组排名]：在每个品牌内排名，保留在原表中
df['品牌内排名'] = df.groupby('品牌')['销售额'].transform(lambda x: x.rank(method='dense', ascending=False))

# 4. 💰 [累计求和]：按时间顺序，从第 1 行开始累加销售额
df = df.sort_values('日期').reset_index(drop=True)
df['累计销售额'] = df['销售额'].cumsum()

### 【SQL 对照】窗口函数

```sql
-- SQL 版本
SELECT 
    *,
    SUM(销售额) OVER() AS 全市场销售额,
    AVG(价格) OVER(PARTITION BY 品牌) AS 本品牌均价,
    DENSE_RANK() OVER(PARTITION BY 品牌 ORDER BY 销售额 DESC) AS 品牌内排名,
    SUM(销售额) OVER(ORDER BY 日期) AS 累计销售额
FROM df;
```

**Pandas 对应**：`transform` 就是你的「窗口函数」

---
## 4️⃣ 文本处理：`.str` 大法

> **大白话口诀**：不要用 `for` 循环和 `apply` 去一行行查字！用 Pandas 自带的 `.str` 文本方法，速度快 100 倍。这对应 SQL 里的 `LIKE '%关键词%'`。

In [ ]:
# 1. 🔍 [检查包含]：判断"显卡名"是否包含"RTX"或"GDDR"
keywords = 'RTX|GDDR'
df['含新显卡'] = df['显卡名'].str.upper().str.contains(keywords, na=False).astype(int)

# 2. ✂️ [提取子串]：从"北京市朝阳区"提取前 2 个字（"北京"）
df['城市代码'] = df['地址'].str[:2]  # 切前 2 个字

# 3. 🔗 [合并字符串]：把"品牌"和"型号"拼成"完整名"
df['完整名'] = df['品牌'] + '-' + df['型号']

# 4. 🧹 [清理空格和特殊符号]：见前面的"脏字符清理"章节
df['干净名'] = df['原始名'].str.replace(r'[\s,]', '', regex=True).str.strip()

---
## 5️⃣ 时间处理：`.dt` 大法

> **大白话口诀**：老板要按"月"看报表，但你的数据是 `"2026-03-23 15:30:00"` 这种精确到秒的怎么办？
> 不要用 `str` 切割！先用 `pd.to_datetime` 把它变成 Pandas 认识的时间老人，然后用 `.dt.year / .dt.month` 瞬间抽取出你想要的部分来做 `groupby`！

In [ ]:
# 1. 🧙‍♂️ [点石成金：字符串变时间格式]
# 第一步永远是把 "2026-03-23" 这种文本字符串，转化为被 Pandas 认可的时间类型
df['下单时间'] = pd.to_datetime(df['下单时间'])

# 2. ✂️ [切片大师：用 .dt 拿取任意时间颗粒度]
# 前提：你的列已经是 datetime 格式了，然后加上 .dt 就能随便拿
df['年份'] = df['下单时间'].dt.year            # 提取：2026
df['月份'] = df['下单时间'].dt.month           # 提取：3
df['几号'] = df['下单时间'].dt.day             # 提取：23
df['星期几'] = df['下单时间'].dt.dayofweek      # 提取：0 (周一) ~ 6 (周日)
df['只拿日期'] = df['下单时间'].dt.date          # 砍掉时分秒

# 3. 📊 [实战连招]：老板要看每个月的总销售额
df['交易月'] = pd.to_datetime(df['购买时间']).dt.to_period('M')  # 转成"202603"这种格式
monthly_sales = df.groupby('交易月')['金额'].sum()

### 🚨 坑点：datetime 转换的坑

#### 坑点 1：pd.to_datetime 失败，报错"unknown datetime format"

```python
# ❌ 格式混乱（有的"2026-03-23"，有的"2026/3/23"，有的"23-03-2026"）
df['日期'] = pd.to_datetime(df['日期'])  # 报错或结果混乱

# ✅ 指定格式或用 infer_datetime_format
df['日期'] = pd.to_datetime(df['日期'], format='%Y-%m-%d')
# 或
df['日期'] = pd.to_datetime(df['日期'], infer_datetime_format=True)
```

---

#### 坑点 2：dayofweek 的编号容易搞反

```python
# 💡 记住这个：0=周一, 1=周二, ..., 6=周日
# 如果你想要周一=1, 周日=7 的 SQL 风格
df['sql_day_of_week'] = df['日期'].dt.dayofweek + 1
```

---
## 6️⃣ 打标签的终极武器：`np.where` & `np.select`

> **大白话口诀**：当 Pandas 的条件过于复杂，或者需要多档位打标签，用 NumPy 秒杀。这对应 SQL 的 `CASE WHEN`。

In [ ]:
# 1. 🚦 [两档开关：np.where]
# np.where(条件, 是的话填什么, 不是的话填什么)
df['定位'] = np.where(df['价格'] > 8000, '高端', '普通')

# 2. 🚦 [多档开关：np.select] (SQL CASE WHEN 的超级版本)
conditions = [
    (df['价格'] >= 10000), 
    (df['价格'] >= 6000) & (df['价格'] < 10000), 
    (df['价格'] < 6000)
]
choices = ['顶级尊享', '主流配置', '垃圾入门']
df['细分市场'] = np.select(conditions, choices, default='其他未知')

# 【SQL 对比】
# CASE WHEN 价格 >= 10000 THEN '顶级尊享'
#      WHEN 价格 >= 6000 THEN '主流配置'
#      WHEN 价格 < 6000 THEN '垃圾入门'
#      ELSE '其他未知'
# END

### 💡 np.where vs pd.cut vs np.select 的选择

| 方法 | 场景 | 速度 | 易用性 |
|------|------|------|--------|
| `np.where` | 只需要两档分类 | ⭐⭐⭐⭐⭐ 最快 | ⭐⭐⭐⭐⭐ 最简单 |
| `np.select` | 需要 3+ 档分类，条件复杂 | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ |
| `pd.cut` | 连续数值分档，比如按"价格区间"自动分 5 档 | ⭐⭐⭐ | ⭐⭐⭐ |
| `pd.loc + mask` | 多次赋值（很不推荐） | ⭐ 最慢 | ⭐ 最麻烦 |

### 【新增】pd.cut() 连续分箱 - 自动切割数值范围 (2026-05-11)

> **何时用**：需要把连续数值（如价格、温度、距离）按**固定区间**或**等频分档**自动切割成分类。比如把 0-1000 的价格分成 10 档、20 档等。

**核心思路**：
- **定义区间边界** (`bins`)：按什么分割点切割
- **定义区间标签** (`labels`)：每个区间叫什么名字  
- **返回结果**：新列是**分类类型**，可以直接做 `groupby()`

**最常用的写法**：

```python
# 1️⃣ 【标准用法】定义 bins 边界和 labels
bins = [0, 5, 10, 20, 50, 100, 666]
labels = ['0-5', '5-10', '10-20', '20-50', '50-100', '100+']

df['price_range'] = pd.cut(df['price'], bins=bins, labels=labels)

# 现在 df['price_range'] 的值会是：'0-5', '5-10' 等等
# 可以直接拿来做分组统计
price_dist = df.groupby('price_range').size()
```

**参数详解**：

| 参数 | 说明 | 例子 |
|------|------|------|
| `x` | 要切割的列 | `df['price']` |
| `bins` | 切割点，列表形式 | `[0, 5, 10, 20]` 表示 0-5, 5-10, 10-20 |
| `labels` | 区间的名字，和 bins 的区间数必须对应 | `['便宜', '中等', '贵']` |
| `right` | 默认 True，表示区间右边界闭合 | `(0, 5]`、`(5, 10]`；如果 False 则 `[0, 5)`、`[5, 10)` |
| `include_lowest` | 默认 False，最小的边界值是否包含 | 设为 True 防止最小值被丢弃 |

**完整实例（来自电商转化分析）**：

```python
import pandas as pd
import numpy as np

# 示例数据：电商商品表
df = pd.DataFrame({
    'product': ['iPhone', 'OnePlus', 'iPad', 'MacBook', '小米手机', '华为平板'],
    'price': [8999, 3999, 6999, 12999, 1999, 2299],
    'purchase_count': [120, 45, 78, 35, 200, 88]
})

# ✅ 方案：按价格区间分析购买行为
bins = [0, 2000, 5000, 10000, 15000]
labels = ['入门 (<2K)', '中低 (2K-5K)', '中高 (5K-10K)', '高端 (>10K)']

df['price_range'] = pd.cut(df['price'], bins=bins, labels=labels)

# 查看分布
print(df)
print("\n各价格段的销量统计：")
print(df.groupby('price_range')['purchase_count'].agg(['count', 'sum', 'mean']))
```

**进阶：等频分箱（不是固定边界，而是让每个箱里的数据量相等）**：

```python
# ❌ 固定边界（可能某些箱很满，某些很空）
df['price_range'] = pd.cut(df['price'], bins=[0, 1000, 5000, 15000])

# ✅ 等频分箱：把数据平均分成 5 份
df['price_range'] = pd.qcut(df['price'], q=5)
# 这样每个区间里的数据量大约相等（都是 20%）
```

**🚨 坑点与解决方案**：

| 坑点 | 症状 | 解决方案 |
|------|------|--------|
| **边界值不合理** | 报错 "Edge of `bins` is not unique" | 检查 bins 是否有重复，且必须升序排列 |
| **最小值被丢弃** | 最小值显示为 NaN | 添加 `include_lowest=True` 参数 |
| **labels 数量不对** | 报错 `ValueError: the number of bin edges (X) should be` | bins 有 n 个值，labels 应该有 n-1 个 |
| **包含 NaN 值** | NaN 行变成 NaN（未分配到任何箱） | 用 `.fillna()` 填充或预处理 NaN |

**完整代码示例（处理异常情况）**：

```python
# 【安全写法】处理缺失值和边界问题
df['price'] = df['price'].fillna(0)  # 先处理 NaN

bins = [0, 5, 10, 20, 50, 100, float('inf')]  # 用 float('inf') 表示最大值
labels = ['0-5', '5-10', '10-20', '20-50', '50-100', '100+']

df['price_range'] = pd.cut(
    df['price'], 
    bins=bins, 
    labels=labels,
    right=False,  # [0, 5), [5, 10) 左闭右开，更符合习惯
    include_lowest=False  # 因为用了 float('inf')，所以不需要
)

print(df['price_range'].value_counts())
```

**pd.cut vs np.select vs pd.loc 的选择**：

| 方法 | 何时用 | 优缺点 |
|------|--------|--------|
| **pd.cut** | 连续数值按固定区间自动切割 | ✅ 参数少，代码简洁；❌ 只能处理数值，条件必须是区间 |
| **np.select** | 多档分类，条件复杂（可以是任意逻辑） | ✅ 灵活强大；❌ 代码多，条件列表和值列表易出错 |
| **pd.loc** | 需要多次赋值，或条件涉及多列 | ✅ 直观易懂；❌ 代码重复，性能差 |


In [ ]:
# 【实战演示】pd.cut() 的完整应用
import pandas as pd
import numpy as np

# ===== 示例 1：电商商品价格分布 =====
df_products = pd.DataFrame({
    'product_id': [1, 2, 3, 4, 5, 6, 7, 8],
    'name': ['iPhone', 'OnePlus', 'iPad', 'MacBook', '小米手机', '华为平板', 'AirPods', '机械键盘'],
    'price': [8999, 3999, 6999, 12999, 1999, 2299, 1999, 599]
})

print("【原始数据】")
print(df_products)
print()

# 【标准用法】定义 bins 和 labels
bins = [0, 2000, 5000, 10000, 15000]
labels = ['入门(<2K)', '中低(2K-5K)', '中高(5K-10K)', '高端(>10K)']

df_products['price_range'] = pd.cut(df_products['price'], bins=bins, labels=labels)

print("【按价格分箱后的结果】")
print(df_products)
print()

# 查看各价格段的数量分布
price_dist = df_products.groupby('price_range').size()
print("【各价格段商品数量】")
print(price_dist)
print()

# ===== 示例 2：结合实际业务分析 =====
# 模拟电商订单表
df_orders = pd.DataFrame({
    'order_id': range(1, 11),
    'product_id': [1, 3, 2, 4, 5, 1, 6, 3, 2, 4],
    'price': [8999, 6999, 3999, 12999, 1999, 8999, 2299, 6999, 3999, 12999],
    'quantity': [1, 2, 1, 1, 3, 1, 2, 1, 2, 1]
})

# 按价格段分箱
df_orders['price_range'] = pd.cut(
    df_orders['price'], 
    bins=[0, 2000, 5000, 10000, 15000],
    labels=['入门(<2K)', '中低(2K-5K)', '中高(5K-10K)', '高端(>10K)']
)

# 统计每个价格段的订单量和总销售额
print("【各价格段订单统计】")
order_summary = df_orders.groupby('price_range').agg({
    'order_id': 'count',       # 订单数
    'quantity': 'sum',         # 总销量
    'price': 'mean'            # 平均价格
}).rename(columns={
    'order_id': '订单数',
    'quantity': '总销量',
    'price': '平均价格'
})
print(order_summary)
print()

# ===== 示例 3：pd.qcut 等频分箱 =====
print("【等频分箱示例】自动把数据分成 3 等份（每份约 33%）")
prices_large = np.random.normal(5000, 3000, 100)
df_qcut = pd.DataFrame({'price': prices_large})
df_qcut['freq_bin'] = pd.qcut(df_qcut['price'], q=3, labels=['低价', '中价', '高价'])
print(df_qcut['freq_bin'].value_counts().sort_index())
print()

# ===== 示例 4：处理边界问题 =====
print("【安全写法：处理边界和 NaN】")
df_safe = df_orders.copy()
# 如果有超出范围的值，用 float('inf') 处理
bins_safe = [0, 2000, 5000, 10000, float('inf')]
labels_safe = ['入门', '中低', '中高', '高端']
df_safe['price_range_safe'] = pd.cut(df_safe['price'], bins=bins_safe, labels=labels_safe, right=False)
print(df_safe[['price', 'price_range_safe']])


### 【新增实战】三种常用打标方法对比 (2026-05-05)

**场景**：根据价格、内存、品牌等多个维度给产品“打标签”。

**1. ⚡ np.where (两档快切)**
> **何时用**：简单的“是非”判断，速度最快。
```python
# 示例：超过 15000 为高端，否则为常规
df['价格定位'] = np.where(df['售价'] >= 15000, '高端', '常规')
```

**2. 🛠️ df.loc (多条件分档)**
> **何时用**：逻辑比较固定，且需要覆盖多个档位时。
```python
# 核心思路：先给个默认值，再用 .loc 覆盖符合条件的行
df['档位档次'] = '入门级'
df.loc[(df['售价'] >= 8000) & (df['内存'] >= 16), '档位档次'] = '中端'
df.loc[(df['售价'] >= 15000) & (df['内存'] >= 32), '档位档次'] = '高端'
```

**3. 🧠 df.apply (极致复杂逻辑)**
> **何时用**：逻辑非常复杂，涉及 `if-elif-else` 或横跨多列的非数值判断。
```python
def complex_logic(row):
    if row['品牌'] in ['苹果', '华为']:
        return '品牌溢价款'
    elif row['售价'] >= 8000 and row['内存'] >= 32:
        return '正常'
    else:
        return '其他'

# 注意 axis=1 表示按行处理
df['市场标签'] = df.apply(complex_logic, axis=1)
```

**🚨 坑点提示**：
- **loc 的括号**：多条件筛选时，每个条件必须用 `()` 括起来，中间用 `&` (且) 或 `|` (或)。
- **apply 的性能**：数据量超过 100 万行时，`apply` 会明显变慢，建议优先用 `np.select` 或 `np.where`。
- **列名匹配**：打标前先确认列名是否存在，可以使用 `target_col = '列名' if '列名' in df.columns else '备选名'` 进行兼容。

### 【新增】4. 🎯 pd.map() - 字典映射快速转换 (2026-05-06)

> **何时用**：把分类列（如"档位"、"类别"）用字典快速转换成数字系数或新的分类标签。比 `apply()` 快 5 倍，代码更简洁。

**核心思路**：
```python
# 建立字典：原值 → 新值的映射关系
level_map = {'入门级': 0.85, '中端': 0.95, '高端': 1.00}

# 使用 .map() 做映射（注意：只对 Series 有效，不对 DataFrame 有效）
df['修正系数'] = df['档位档次'].map(level_map)
```

**完整实例（来自三创赛预测系统）**：

```python
# 场景：根据产品"档位"，给出价格修正系数
# 需求：入门级产品涨不动（系数 0.85），高端产品随便涨（系数 1.00）

# ✅ 【快速方案】用 map + 字典
c_map = {'入门级': 0.85, '中端': 0.95, '高端': 1.00}
df['修正系数'] = df['档位档次'].map(c_map)
df['修正价格'] = df['预测价'] * df['修正系数']

# ❌ 【低效方案】用 apply（代码多 3 倍，速度慢 5 倍）
df['修正系数'] = df['档位档次'].apply(lambda x: {'入门级': 0.85, '中端': 0.95, '高端': 1.00}.get(x))
```

**🚨 坑点与进阶技巧**：

| 坑点 | 症状 | 解决方案 |
|------|------|--------|
| **NaN 值处理** | 字典里没有的值被映射成 NaN | 用 `.fillna()` 填充，或用 `.get(x, 默认值)` 设定默认 |
| **map 只能用于 Series** | 试图对 DataFrame 用 map 报错 | 改用 `apply()` 或逐列处理 |
| **想保留原值** | 有些值在字典里，有些不在 | 用 `lambda x: level_map.get(x, x)` 保留未映射的值 |

**完整代码示例**：

```python
# 【安全写法】处理可能存在的未知值
level_map = {'入门级': 0.85, '中端': 0.95, '高端': 1.00}

# 方案 1：未知值映射成 NaN（然后用 .fillna() 填充）
df['修正系数'] = df['档位档次'].map(level_map).fillna(0.9)

# 方案 2：未知值映射成指定默认值
df['修正系数'] = df['档位档次'].apply(lambda x: level_map.get(x, 0.9))

# 【性能对比】
# map 速度：⭐⭐⭐⭐⭐ 最快（仅用于简单字典映射）
# apply 速度：⭐⭐⭐ （能处理复杂逻辑）
```

**适用场景**：
- ✅ 颜色名 → 颜色代码（"红" → "#FF0000"）
- ✅ 品牌简写 → 品牌全名（"AAPL" → "Apple"）
- ✅ 等级 → 系数（"低" → 0.8，"高" → 1.2）
- ✅ 分类 → 权重（"VIP" → 100，"普通" → 1）

In [ ]:
# 【实战演示】map 的完整应用流程
import pandas as pd
import numpy as np

# 模拟数据：电脑产品表
df = pd.DataFrame({
    '产品': ['笔记本A', '笔记本B', '台式机C', '笔记本D'],
    '档位档次': ['入门级', '中端', '高端', '中端'],
    '预测价': [5000, 8000, 15000, 9000]
})

print("【原始数据】")
print(df)
print()

# ✅ 方案 1：基础 map（最简单快速）
c_map = {'入门级': 0.85, '中端': 0.95, '高端': 1.00}
df['修正系数'] = df['档位档次'].map(c_map)
df['修正价格'] = df['预测价'] * df['修正系数']

print("【使用 map 后的结果】")
print(df)
print()

# ✅ 方案 2：处理未知值（安全写法）
df_with_unknown = df.copy()
df_with_unknown.loc[1, '档位档次'] = '未知等级'  # 故意加个字典里没有的值

df_with_unknown['修正系数_安全'] = df_with_unknown['档位档次'].map(c_map).fillna(0.9)
print("【处理未知值后的结果（用 fillna 填充）】")
print(df_with_unknown)
print()

# ⚡ 对比：map vs apply 的性能
import time

# 生成大数据集
large_df = pd.DataFrame({
    '档位': np.random.choice(['入门级', '中端', '高端'], 100000)
})

# 用 map（快）
start = time.time()
result1 = large_df['档位'].map(c_map)
time_map = time.time() - start

# 用 apply（慢）
start = time.time()
result2 = large_df['档位'].apply(lambda x: c_map.get(x))
time_apply = time.time() - start

print(f"【性能对比】")
print(f"  map 耗时：{time_map:.6f} 秒")
print(f"  apply 耗时：{time_apply:.6f} 秒")
print(f"  速度提升：{time_apply/time_map:.1f}x")

---
## 【连招】常用数据分析链路

### 场景：我要分析"各品牌在不同月份的销售情况"

```python
# 第 1 步：准备数据
df['交易月'] = pd.to_datetime(df['日期']).dt.to_period('M')

# 第 2 步：分组统计
summary = df.groupby(['品牌', '交易月']).agg({
    '销售额': 'sum',
    '销售量': 'mean'
}).reset_index()

# 第 3 步：广播计算（我比平均高多少）
summary['品牌月均'] = summary.groupby('品牌')['销售额'].transform('mean')
summary['对比'] = summary['销售额'] - summary['品牌月均']

# 第 4 步：打标签（是否超过平均）
summary['达标'] = np.where(summary['对比'] > 0, '✓ 超额', '✗ 未达')

# 第 5 步：排序查看
result = summary.sort_values('销售额', ascending=False).head(10)
```